# Face model v2 — round 5: FER+ labels, checkpoint chosen by FER+ validation
Round 4 showed pure FER+ soft labels give val Macro-F1 0.70 against FER+ majority labels (res112 baseline 0.58, v1 0.55).
This round picks the checkpoint by the FER+ score (`--select ferplus`) and repeats res96 with 3 seeds to measure noise.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r5.zip`, the FER2013 zip, `out/face_v2_results_r4.zip`. Runtime → T4 GPU.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, zipfile
D='/content/drive/MyDrive/face-v2'
os.makedirs('/content/Main_demo', exist_ok=True)
zipfile.ZipFile(f'{D}/Face-Lab-code-r5.zip').extractall('/content/Main_demo')
fer = [p for p in glob.glob(f'{D}/*.zip') if 'Face-Lab' not in p][0]
zipfile.ZipFile(fer).extractall('/content/Main_demo/FER2013-Data')
zipfile.ZipFile(f'{D}/out/face_v2_results_r4.zip').extractall('/content/Main_demo/Face-Lab')   # rounds 1-4 results/ for the table
for s in ('train','test'):
    print(s, sum(len(f) for _,_,f in os.walk(f'/content/Main_demo/FER2013-Data/{s}')))   # expect 28709 / 7178
%cd /content/Main_demo/Face-Lab
print(sorted(os.listdir('results')))

In [ ]:
BASE = '--arch efficientnet_b0 --epochs 20 --head-epochs 1 --unfreeze 0.0 --split-seed 42'
FP = '--ferplus 1 --smooth 0 --select ferplus'
EXPS = {
  'v2_fps_res96_s42':  f'--res 96 {FP} --seed 42',
  'v2_fps_res96_s1':   f'--res 96 {FP} --seed 1',
  'v2_fps_res96_s2':   f'--res 96 {FP} --seed 2',
  'v2_fps_res112_s42': f'--res 112 {FP} --seed 42',
}
for tag, extra in EXPS.items():
    cmd = f'python train.py {BASE} {extra} --tag {tag}'
    print('>>', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    lines = r.stdout.strip().splitlines()
    print(chr(10).join(lines[-3:]) if lines else '', r.stderr[-800:] if r.returncode else '', flush=True)


In [ ]:
import json, glob
rows = {}
for f in sorted(glob.glob('results/v2_fp*.json')):
    r = json.load(open(f)); v = r['val']; fp = r['val_ferplus']; pc = fp['per_class']
    rows[r['tag']] = (fp['macro_f1'], fp['accuracy'], v['macro_f1'], pc['fear']['f1'], pc['disgust']['f1'], pc['sad']['f1'], r['best_epoch'], r['train_minutes'])
print('FER+ val reference: v1 0.553, res112 0.582 (both trained on FER2013 labels; no neutral prior)')
print('run'.ljust(20), 'fp_F1   fp_acc  fer13_F1  fear   disg   sad    ep  min')
for t, (f1, ac, f13, fe, di, sa, ep, mn) in sorted(rows.items(), key=lambda kv: -kv[1][0]):
    print(t.ljust(20), '%.4f  %.4f  %.4f   %.3f  %.3f  %.3f  %3d  %4.1f' % (f1, ac, f13, fe, di, sa, ep, mn))

In [ ]:
OUT='/content/drive/MyDrive/face-v2/out'
shutil.make_archive('/content/face_v2_results_r5','zip','/content/Main_demo/Face-Lab','results')
shutil.copy('/content/face_v2_results_r5.zip', OUT)
for t in EXPS:
    if os.path.exists(f'runs/{t}/best.pt'):
        os.makedirs(f'{OUT}/{t}', exist_ok=True)
        b = open(f'runs/{t}/best.pt','rb').read()
        for i in range(0, len(b), 9000000):                     # parts < 10 MB so they can be pulled back to the laptop
            open(f'{OUT}/{t}/best.pt.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(sorted(os.listdir(OUT)))